# 🛰️ Projeto OrbitalCore: Sistema Inteligente de Telemetria e Monitoramento de Satélites
## Disciplina: Prompt and Artificial Intelligence (PAI)

---

### 👥 Estrutura do Grupo e Divisão de Escopo

**Integrante 1: Isaías Hörlle Sobral | RM: 568990**  

**Integrante 2: Leandro Cavaccini Brito | RM: 570556** 

**Integrante 3: Lucas Dorice Dos Santos | RM: 568592**

---

### 📝 Contextualização do Projeto: O Paradigma dos Data Centers Espaciais

O **OrbitalCore v1.0** foi projetado sob o conceito inovador de um **Data Center Espacial** (*Space Data Center*), aplicando a arquitetura de *Orbital Edge Computing* (Computação de Borda em Órbita). Em vez de saturar a largura de banda transmitindo volumes massivos de dados brutos para as estações de solo, o microssatélite processa e analisa as informações diretamente no espaço. No entanto, operar servidores de alta performance em Órbita Baixa Terrestre (LEO) impõe desafios físicos severos, como ciclagens térmicas violentas (da radiação solar direta ao congelamento na zona de eclipse), ausência de convecção natural para dissipar o calor dos processadores e o risco constante de microimpactos por detritos espaciais.

Para garantir a salvaguarda e a resiliência desse hardware crítico, este componente de **Prompt and Artificial Intelligence (PAI)** atua como um **Engenheiro de Voo Autônomo**. Utilizando o modelo *open-source* **Qwen 2.5 (1.5B-Instruct)** — otimizado via **quantização em Float16 (FP16)** para viabilizar a execução local e embarcada —, o script consome strings JSON dinâmicas de telemetria (temperatura, luminosidade e vibração). Em milissegundos e de forma 100% offline (sem dependência de APIs externas), a IA correlaciona as leituras dos sensores com o cenário orbital atual, diagnosticando anomalias estruturais ou térmicas e gerando planos de ação imediatos em linguagem natural para mitigar riscos catastróficos no Data Center Espacial.

In [ ]:
print("Instalando dependências...")
!pip install -q transformers accelerate
print("✅ SISTEMA PRONTO PARA CARREGAMENTO!")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from huggingface_hub.utils import logging as hf_logging
from transformers import logging as tf_logging
hf_logging.set_verbosity_error()
tf_logging.set_verbosity_error()

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

print("Carregando o assistente na GPU (Float16 Nativo)...")

tokenizer = AutoTokenizer.from_pretrained(model_id, token=False)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    token=False
)

gerador = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("🚀 ASSISTENTE ORBITALCORE PRONTO E OTIMIZADO!")

In [ ]:
import json
import random

# 1. Simulação Dinâmica dos Cenários Orbitais
cenarios = [
    {"nome": "Exposição Solar Direta", "desc": "Satélite sob radiação solar intensa, painéis térmicos em carga máxima de dissipação."},
    {"nome": "Zona de Eclipse Terrestre", "desc": "Satélite na sombra da Terra. Resfriamento severo de hardware e baterias alimentando o core."},
    {"nome": "Cinturão de Detritos / Manobra Ativa", "desc": "Passagem por zona de risco ou acionamento de propulsores para correção orbital."},
    {"nome": "Órbita Estável Padrão", "desc": "Condições nominais de operação em órbita baixa (LEO), sem alertas externos."}
]

cenario_atual = random.choice(cenarios)

# Geração de dados coerentes com base nos sensores do projeto (DHT22, LDR e MPU6050) + Status da Missão
if cenario_atual["nome"] == "Exposição Solar Direta":
    temp, lux, vib = random.uniform(78.0, 96.0), random.uniform(55000, 75000), random.uniform(0.02, 0.07)
    status_missao = "Em Atenção"
elif cenario_atual["nome"] == "Zona de Eclipse Terrestre":
    temp, lux, vib = random.uniform(-35.0, -5.0), 0.0, random.uniform(0.01, 0.04)
    status_missao = "Em Atenção"
elif cenario_atual["nome"] == "Cinturão de Detritos / Manobra Ativa":
    temp, lux, vib = random.uniform(22.0, 32.0), random.uniform(15000, 35000), random.uniform(0.55, 1.45)
    status_missao = "Crítica"
else: # Órbita Estável Padrão
    temp, lux, vib = random.uniform(18.0, 26.0), random.uniform(25000, 45000), random.uniform(0.01, 0.03)
    status_missao = "Normal ou Estável"

telemetria = {
    "cenario_orbital": cenario_atual["nome"],
    "status_missao": status_missao,
    "descricao_contexto": cenario_atual["desc"],
    "temperatura_dht22_c": round(temp, 2),
    "luminosidade_ldr_lux": round(lux, 2),
    "vibracao_mpu6050_g": round(vib, 2)
}

string_json = json.dumps(telemetria, indent=4, ensure_ascii=False)

print("📡 [TELEMETRIA CAPTURADA - SPACE DATA CENTER]:")
print(string_json)
print("-" * 60)

# 2. Engenharia de Prompt Estruturada
system_prompt = (
    "Você é o Engenheiro de Voo Autônomo do Data Center Espacial OrbitalCore v1.0.\n"
    "Analise o JSON de telemetria recebido e forneça um parecer técnico rápido em português.\n"
    "Se houver anomalias, crie um plano de ação imediato para preservar os servidores. Seja formal e objetivo."
)

mensagens = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Analise esta telemetria:\n\n{string_json}"}
]
prompt_formatado = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)

# 3. Geração do Diagnóstico Técnico
print("🧠 IA ORBITALCORE PROCESSANDO EM EDGE COMPUTING...")
saida = gerador(prompt_formatado, max_new_tokens=300, do_sample=True, temperature=0.7, top_p=0.9)

print("\n📋 [RELATÓRIO DE SALVAGUARDA DE MISSÃO]")
print(saida[0]['generated_text'].split("<|im_start|>assistant")[-1].strip())